# Federated Digital Twins: Automated Full Real-Data Validation (All 3 Datasets)

**Run this top to bottom, once.** It downloads and runs all 15 experiment sections against CICIoT2023, Edge-IIoTset, and N-BaIoT in turn, saves every result as a structured file (CSV/JSON/PNG, not printed text), and ends by zipping everything into one `results_bundle.zip` for you to download and send back.

**What's different from the previous notebook:**
- No manual `DATASET_CHOICE` editing or re-running -- one loop drives all three datasets.
- Every experiment is a function that takes the dataset as a parameter and returns/saves its results, instead of top-level cells that assume a specific dataset is already loaded.
- If a dataset fails to load (e.g.\ Edge-IIoTset's column names don't match, since that loader has never been run against the real file before) or one experiment errors out, it's logged to `run_log.txt` and the run **continues** with the next experiment/dataset rather than stopping. You get everything that succeeded, plus a clear record of what didn't.
- The seed-resampling bug from the previous notebook (each seed re-sampling a fresh slice of the raw CSV, rather than reusing one fixed sample) is fixed here: each dataset's data is loaded **once** and cached; only downstream training/attack/partition randomness varies by seed.

**Before running:** upload `kaggle.json` to `/content` via the Colab file browser. Everything else is automatic.

**Expect this to take a while** -- multiple hours for all three datasets at the default settings. It's safe to let it run in the background; if Colab disconnects partway, whatever finished is already saved to disk (results are written incrementally, not held in memory until the end), though you'll need to re-run from the top to regenerate what didn't finish. Consider Colab Pro / background execution if available.

## 0. Configuration

In [ ]:
import warnings, json, copy, os, glob, time, shutil, zipfile, traceback
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from collections import Counter
import shap

# ============ WHICH DATASETS -- all three, automatically ============
DATASETS = ['ciciot', 'edgeiiot', 'nbaiot']
CICIOT_DATA_DIR = 'data/ciciot2023'
EDGEIIOT_DATA_DIR = 'data/edgeiiotset'
NBAIOT_DATA_DIR = 'data/nbaiot'
DATA_DIRS = {'ciciot': CICIOT_DATA_DIR, 'edgeiiot': EDGEIIOT_DATA_DIR, 'nbaiot': NBAIOT_DATA_DIR}
KAGGLE_SLUGS = {
    'ciciot': 'akashdogra/ciciot23csv',
    'edgeiiot': 'mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot',
    'nbaiot': 'mkashifn/nbaiot-dataset',
}
RESULTS_DIR = 'results'

# ============ RUNTIME BUDGET ============
QUICK_MODE = True                 # True: fewer seeds + smaller subsample for the expensive sections
                                   # (baselines, ablation, adaptive, sensitivity). False: full 3-seed runs.
MAX_ROWS_FULL = 1_500_000         # used for the single-attacker and headline severity-sweep sections
MAX_ROWS_QUICK = 80_000          # used for the expensive multi-config sections when QUICK_MODE=True
SEEDS_FULL = [1, 7, 21, 42, 100]
SEEDS_QUICK = [1, 7, 21, 42, 100]

# ============ EXPERIMENT HYPERPARAMETERS (fixed across every dataset/section) ============
N_CLIENTS = 8               # 9 automatically for nbaiot (its natural device count)
ROUNDS = 15
LOCAL_EPOCHS = 2
LR = 5e-3
FEDPROX_MU = 0.0
POISON_CLIENT = 0
VETO_EPSILON = 0.0
VETO_K = 4.0
WEIGHT_CAP = 20.0

os.makedirs(RESULTS_DIR, exist_ok=True)
print('Config loaded. Datasets to run:', DATASETS, ' | QUICK_MODE:', QUICK_MODE)

## 1. Data loaders (unchanged from the tested single-dataset notebook)
`load_real_ciciot` has been run end-to-end against the actual downloaded dataset. `load_real_edgeiiot` and `load_real_nbaiot` are schema-correct by construction (tested against mock file/folder structures, including the N-BaIoT device-name bug fix) but have not been run against the real files before -- if either fails, the error is caught and logged (Section 3) rather than stopping the whole run.

In [ ]:
def load_csvs_streamed(files, sample_frac=None, max_rows=None, chunksize=100_000, seed=42, verbose=True):
    '''Reads CSVs in bounded-memory chunks, sub-sampling as it goes, so the full
    raw dataset is never materialised in RAM at once -- tested at 3.6GB multi-file
    input, ~227MB peak RSS. Do not replace with pd.concat([pd.read_csv(f) for f in files]).'''
    if sample_frac is None and max_rows is None:
        raise ValueError('provide sample_frac or max_rows')
    if sample_frac is None:
        total_bytes = sum(os.path.getsize(f) for f in files)
        approx_total_rows = total_bytes / 200
        sample_frac = min(1.0, (max_rows / approx_total_rows) * 1.5)
    parts, total_kept = [], 0
    for fi, f in enumerate(files):
        for chunk in pd.read_csv(f, chunksize=chunksize):
            keep = chunk.sample(frac=min(sample_frac, 1.0), random_state=seed + fi)
            if len(keep) > 0:
                parts.append(keep); total_kept += len(keep)
            if max_rows and total_kept >= max_rows: break
        if verbose: print(f'  read {os.path.basename(f)}  (running sample: {total_kept:,} rows)')
        if max_rows and total_kept >= max_rows: break
    if not parts:
        raise ValueError('no rows read -- check file paths/format')
    df = pd.concat(parts, ignore_index=True)
    if max_rows and len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=seed).reset_index(drop=True)
    return df

In [ ]:
def group_attack_category(raw_label):
    '''Confirmed working against real CICIoT2023's actual label column.'''
    s = str(raw_label).lower()
    if 'benign' in s: return 'Benign'
    if 'ddos' in s: return 'DDoS'
    if 'dos' in s: return 'DoS'
    if 'recon' in s: return 'Recon'
    if 'mirai' in s: return 'Mirai'
    if any(k in s for k in ['mitm', 'spoof', 'arp']): return 'Spoofing'
    if any(k in s for k in ['bruteforce', 'brute_force', 'dictionary']): return 'BruteForce'
    if any(k in s for k in ['sql', 'xss', 'upload', 'command', 'backdoor', 'web']): return 'Web-based'
    return 'Other'


def load_real_ciciot(data_dir=None, label_col='label', device_col=None, max_rows=1_500_000, seed=42):
    '''Confirmed working against the real, downloaded CICIoT2023 CSV(s). No device/
    source-IP column survives in this export -- device_id falls back to pseudo-random
    assignment (dev_000-style); this is a known, disclosed limitation, not a bug.'''
    data_dir = data_dir or CICIOT_DATA_DIR
    files = sorted(glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True))
    if not files:
        raise FileNotFoundError(f'No CSVs under {data_dir} -- run the download cell in Section 2 first')
    df = load_csvs_streamed(files, max_rows=max_rows, seed=seed)
    df.columns = [c.strip() for c in df.columns]
    if label_col not in df.columns:
        raise KeyError(f'label_col={label_col!r} not found; available columns: {list(df.columns)[:10]}...')
    df['label'] = df[label_col].apply(group_attack_category)
    df = df[df['label'] != 'Other'].reset_index(drop=True)
    rng = np.random.default_rng(seed)
    if device_col and device_col in df.columns:
        df['device_id'] = df[device_col].astype(str)
    else:
        df['device_id'] = [f'dev_{i:03d}' for i in rng.integers(0, 40, size=len(df))]
    feature_names = [c for c in df.columns if c not in ('label', 'device_id', label_col, device_col)
                     and pd.api.types.is_numeric_dtype(df[c])]
    return df.reset_index(drop=True), feature_names

In [ ]:
def group_edgeiiot_category(raw_label):
    s = str(raw_label).lower()
    if 'normal' in s: return 'Normal'
    if 'ddos' in s: return 'DDoS'
    if any(k in s for k in ['scan', 'fingerprint']): return 'InfoGathering'
    if 'mitm' in s: return 'MITM'
    if any(k in s for k in ['sql', 'xss', 'upload', 'backdoor', 'password']): return 'Injection'
    if 'ransomware' in s: return 'Malware'
    return 'Other'


def load_real_edgeiiot(data_dir=None, label_col='Attack_type', device_col=None, max_rows=1_000_000, seed=42):
    '''Official Kaggle mirror: mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot.
    Downloads sometimes bundle multiple CSV variants with DIFFERENT schemas; this keeps
    only files matching the majority schema (tested against a mock mixed-schema folder).
    No documented per-record device-ID column -- pseudo-assigns devices unless device_col
    points at a source-IP-like field that exists in your download.
    RUN THE DIAGNOSTIC CELL BELOW FIRST -- label_col=\'Attack_type\' is documented but unverified
    against a real download by us; check it against your actual file columns.'''
    data_dir = data_dir or EDGEIIOT_DATA_DIR
    files = sorted(glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True))
    if not files:
        raise FileNotFoundError(f'No CSVs under {data_dir} -- run the download cell in Section 2 first')
    headers = {f: tuple(pd.read_csv(f, nrows=0).columns) for f in files}
    majority_schema = Counter(headers.values()).most_common(1)[0][0]
    good_files = [f for f in files if headers[f] == majority_schema]
    skipped = [f for f in files if f not in good_files]
    if skipped:
        print(f'NOTE: skipped {len(skipped)} file(s) with a different schema than the majority: {skipped}')
    df = load_csvs_streamed(good_files, max_rows=max_rows, seed=seed)
    df.columns = [c.strip() for c in df.columns]
    if label_col not in df.columns:
        raise KeyError(f'{label_col!r} not found; available columns: {list(df.columns)[:15]}... '
                       f'-- run the Section 1b diagnostic and fix label_col above')
    df['label'] = df[label_col].apply(group_edgeiiot_category)
    df = df[df['label'] != 'Other'].reset_index(drop=True)
    rng = np.random.default_rng(seed)
    if device_col and device_col in df.columns:
        df['device_id'] = df[device_col].astype(str)
    else:
        df['device_id'] = [f'dev_{i:03d}' for i in rng.integers(0, 20, size=len(df))]
    # Edge-IIoTset's real schema carries BOTH 'Attack_type' (multi-class, our label_col) and a
    # separate binary 'Attack_label' (normal=0/attack=1) column. The latter is a near-duplicate
    # of the target and was previously left IN the feature set by accident -- confirmed via a
    # real run where it dominated SHAP importance (0.33 vs 0.10 for the next feature). Excluded
    # explicitly here, along with any other obviously label-derived column.
    _leaky = {'attack_label', 'attack_type', str(label_col).lower()}
    feature_names = [c for c in df.columns if c not in ('label', 'device_id', label_col)
                     and c.lower() not in _leaky
                     and pd.api.types.is_numeric_dtype(df[c])]
    return df.reset_index(drop=True), feature_names

In [ ]:
def load_real_nbaiot(data_dir=None, max_rows=1_000_000, seed=42, chunksize=100_000):
    '''Official mirrors (Kaggle mkashifn/nbaiot-dataset, or UCI ML Repository) distribute
    this as SEPARATE FILES PER REAL DEVICE (9 devices) -- the only one of the three
    datasets with a genuine one-twin-per-real-device story. Device name is parsed from
    each file's path: attack files are commonly nested one level deeper than benign files
    (DeviceName/gafgyt_attacks/combo.csv vs DeviceName/benign_traffic.csv) -- naively
    taking the immediate parent mislabels every attack row with the attack-folder name
    instead of the device. This was a real bug caught by testing against a mock folder
    tree; the fix (walk up one more level for attack files) is applied below.
    RUN THE DIAGNOSTIC CELL BELOW FIRST -- this loader has not yet been run against the
    real download; check the printed cross-tab shows every device under all 3 labels.'''
    data_dir = data_dir or NBAIOT_DATA_DIR
    files = sorted(glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True))
    if not files:
        raise FileNotFoundError(f'No CSVs under {data_dir} -- run the download cell in Section 2 first')
    import re
    labelled_files = []
    parent_dirs_seen, filenames_seen = set(), []
    for f in files:
        fname = os.path.basename(f).lower()
        if 'benign' in fname: label = 'Benign'
        elif 'mirai' in fname: label = 'Mirai'
        elif any(k in fname for k in ['gafgyt', 'bashlite']): label = 'Gafgyt'
        else: continue
        filenames_seen.append(os.path.basename(f))
        # Primary: the standard N-BaIoT/UCI distribution is a FLAT directory of files named
        # like '1.benign.csv', '7.mirai.scan.csv' -- device is the leading number (1-9).
        m = re.match(r'^(\d+)\.', os.path.basename(f))
        if m:
            device = f'device_{m.group(1)}'
        else:
            # Fallback: a nested per-device-folder layout (DeviceName/benign_traffic.csv,
            # DeviceName/mirai_attacks/scan.csv) -- attack files one level deeper than benign.
            parent = os.path.basename(os.path.dirname(f))
            parent_dirs_seen.add(parent)
            if any(k in parent.lower() for k in ['mirai', 'gafgyt', 'bashlite', 'attack']):
                device = os.path.basename(os.path.dirname(os.path.dirname(f))) or parent
            else:
                device = parent or 'unknown_device'
        labelled_files.append((f, label, device))
    if not labelled_files:
        raise ValueError('no files matched a benign/mirai/gafgyt path pattern -- run the Section 1b '
                          'diagnostic and check the printed file paths against the keywords above')
    n_unique_devices = len(set(d for _, _, d in labelled_files))
    if n_unique_devices < 2:
        raise ValueError(
            f'device parsing produced only {n_unique_devices} unique device(s) from '
            f'{len(labelled_files)} files -- this means neither the filename-number pattern nor the '
            f'folder-nesting fallback matched your actual download layout. Sample filenames seen: '
            f'{filenames_seen[:8]}. Sample parent folders seen: {list(parent_dirs_seen)[:8]}. '
            f'Inspect these and fix the parsing logic above before trusting any downstream result.')
    per_file_budget = max(1, max_rows // len(labelled_files))
    parts = []
    for f, label, device in labelled_files:
        got = 0
        for chunk in pd.read_csv(f, chunksize=chunksize):
            keep = chunk.sample(n=min(per_file_budget - got, len(chunk)), random_state=seed) if got < per_file_budget else chunk.iloc[0:0]
            if len(keep) > 0:
                keep = keep.copy(); keep['label'] = label; keep['device_id'] = device
                parts.append(keep); got += len(keep)
            if got >= per_file_budget: break
    df = pd.concat(parts, ignore_index=True)
    feature_names = [c for c in df.columns if c not in ('label', 'device_id')
                     and pd.api.types.is_numeric_dtype(df[c])]
    return df.reset_index(drop=True), feature_names

## 2. Kaggle auth + automated per-dataset download and load

In [ ]:
# Uploads kaggle.json interactively -- no need to find the file browser panel yourself.
# Get kaggle.json from: kaggle.com -> your profile picture -> Settings -> API -> Create New Token
# (downloads a kaggle.json file to your computer). Running this cell will pop up a file picker.
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Select your kaggle.json file in the picker that appears below:')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        print(f"WARNING: expected a file named 'kaggle.json', got: {list(uploaded.keys())}")
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('kaggle.json installed')
else:
    print('kaggle.json already installed from a previous run in this session')


### Verify kaggle CLI works before the long run (recommended)
This should print a short table of matching datasets. If it prints an error instead (auth failure, command not found, etc.), fix it here -- this takes seconds to check and saves hours of wasted runtime.

In [ ]:
# Quick sanity check -- confirms the kaggle CLI actually works BEFORE the long run starts.
# If this prints an error instead of a dataset list, fix that now rather than discovering
# it 3 hours into Section 17 with an empty results_bundle.zip.
try:
    ip = get_ipython()
    ip.system('kaggle datasets list -s ciciot --max-size 1')
except Exception as e:
    print('kaggle CLI check raised:', e)

In [ ]:
def ensure_downloaded(dataset):
    '''Idempotent: skips the download if CSVs already exist under this dataset's folder.
    Uses get_ipython().system() (not os.system()) -- this is the same mechanism the
    single-dataset notebook used and that you confirmed works; os.system() does not
    reliably inherit Colab's PATH/environment for the kaggle CLI and fails silently
    (non-zero exit code, no exception), which is almost certainly why the first version
    of this automated notebook produced an empty results bundle.'''
    data_dir = DATA_DIRS[dataset]
    existing = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
    if existing:
        print(f'[{dataset}] {len(existing)} CSV(s) already present, skipping download')
        return True
    os.makedirs(data_dir, exist_ok=True)
    cmd = f'kaggle datasets download -d {KAGGLE_SLUGS[dataset]} -p {data_dir} --unzip'
    print(f'[{dataset}] running: {cmd}')
    try:
        ip = get_ipython()
        ip.system(cmd)
    except NameError:
        # not running inside IPython/Colab (e.g. a plain Python check) -- fall back, though
        # this path is not expected to work reliably for the kaggle CLI specifically
        os.system(cmd)
    ok = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
    print(f'[{dataset}] download {"OK" if ok else "FAILED"}, {len(ok)} CSV(s) found')
    if not ok:
        print(f'[{dataset}] TROUBLESHOOTING: run !kaggle datasets download -d {KAGGLE_SLUGS[dataset]} -p {data_dir} --unzip')
        print(f'[{dataset}]   directly in its own cell to see the actual kaggle CLI error output.')
    return bool(ok)


_LOAD_FN = {'ciciot': load_real_ciciot, 'edgeiiot': load_real_edgeiiot, 'nbaiot': load_real_nbaiot}
_DATASET_CACHE = {}

def get_dataset(dataset, max_rows):
    '''Loads once per (dataset, max_rows) and caches -- this is the fix for the seed-
    resampling issue: every seed downstream reuses this exact same sample.
    Passes data_dir explicitly (from DATA_DIRS) so a missing/late Section 0 
    global (e.g. after a runtime restart) can't cause a NameError here.'''
    key = (dataset, max_rows)
    if key not in _DATASET_CACHE:
        print(f'[{dataset}] loading (max_rows={max_rows:,}) -- one-time load, cached for all seeds')
        t0 = time.time()
        df, feats = _LOAD_FN[dataset](data_dir=DATA_DIRS[dataset], max_rows=max_rows, seed=0)
        print(f'[{dataset}] loaded {df.shape} in {time.time()-t0:.0f}s')
        _DATASET_CACHE[key] = (df, feats)
    return _DATASET_CACHE[key]

## 3. Non-IID partitioning + the per-seed data helper
`get_clients_for_seed` is the fixed version of the earlier `build_data_for_seed`: it calls the cached `get_dataset` (Section 2) instead of re-loading/re-sampling the raw CSV, so every seed downstream trains/attacks/partitions differently but starts from the identical underlying data sample.

In [ ]:
def make_clients(df, feature_names, label_col, device_col, n_clients, shadow_frac=0.25, seed=0):
    rng = np.random.default_rng(seed)
    devices = sorted(df[device_col].unique())
    rng.shuffle(devices)
    groups = np.array_split(devices, n_clients)
    clients = []
    for grp in groups:
        sub = df[df[device_col].isin(grp)]
        X = sub[feature_names].values.astype(np.float32)
        y = sub[label_col].values
        if len(np.unique(y)) > 1 and min(np.bincount(y)) >= 2:
            Xtr, Xsh, ytr, ysh = train_test_split(X, y, test_size=shadow_frac, stratify=y, random_state=seed)
        else:
            Xtr, Xsh, ytr, ysh = train_test_split(X, y, test_size=shadow_frac, random_state=seed)
        clients.append({'devices': list(grp), 'X_train': Xtr, 'y_train': ytr, 'X_shadow': Xsh, 'y_shadow': ysh})
    return clients


def get_clients_for_seed(dataset, seed, max_rows):
    '''Fixed data sample (from the Section 2 cache) + seed-varying split/partition.
    Returns (clients, feature_names, n_classes, Xte, yte, label_encoder).'''
    df_full, feats = get_dataset(dataset, max_rows)
    d = df_full.copy()
    le = LabelEncoder().fit(d['label']); d['label_enc'] = le.transform(d['label'])
    nc = d['label_enc'].nunique()
    nclients = d['device_id'].nunique() if dataset == 'nbaiot' else N_CLIENTS
    dtr, dte = train_test_split(d, test_size=0.2, stratify=d['label_enc'], random_state=seed)
    sc = StandardScaler().fit(dtr[feats])
    dtr = dtr.copy(); dte = dte.copy()
    dtr[feats] = sc.transform(dtr[feats]); dte[feats] = sc.transform(dte[feats])
    Xte = dte[feats].values.astype(np.float32); yte = dte['label_enc'].values
    clients = make_clients(dtr, feats, 'label_enc', 'device_id', nclients, seed=seed)
    return clients, feats, nc, Xte, yte, le

## 4. Model

In [ ]:
class TwinMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(64, 32)):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.1)]
            prev = h
        layers += [nn.Linear(prev, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def get_state(model): return copy.deepcopy(model.state_dict())
def set_state(model, state): model.load_state_dict(copy.deepcopy(state))

def average_states(states, weights):
    total = sum(weights)
    avg = copy.deepcopy(states[0])
    for k in avg.keys():
        avg[k] = torch.zeros_like(avg[k], dtype=torch.float32)
    for state, w in zip(states, weights):
        for k in avg.keys():
            avg[k] += state[k].float() * (w / total)
    return avg

def state_to_vec(state):
    return torch.cat([v.float().flatten() for v in state.values()])

def vec_to_state_like(vec, template_state):
    new_state = {}; idx = 0
    for k, v in template_state.items():
        n = v.numel()
        new_state[k] = vec[idx:idx+n].reshape(v.shape).clone()
        idx += n
    return new_state

## 5. Federated core: local training + the digital-twin validation gate

**Class weighting is capped, not raw.** Real CICIoT2023's inverse-frequency class weights hit ~740x for the rarest classes -- left uncapped, this was verified to destabilise training badly enough that a fully-poisoned twin's update could look *better* than honest twins' updates (the whole ablation inverted). `WEIGHT_CAP` (Section 0) fixes this.

**The gate**: each twin evaluates its candidate update's cross-entropy loss on its own held-out shadow split, before vs. after local training. A candidate is vetoed -- excluded from that round's aggregate -- if it makes shadow loss meaningfully worse in absolute terms (> `veto_epsilon`), *or* if its loss change is a strong outlier relative to its peers that round (peer-relative MAD test, `veto_k` MADs above the median, with a floor `sigma_min` on the MAD itself to avoid a degenerate over-sensitive threshold when honest twins happen to agree almost exactly).

In [ ]:
def to_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

def compute_class_weights(y, n_classes, cap=None):
    cap = WEIGHT_CAP if cap is None else cap
    counts = np.bincount(y, minlength=n_classes).astype(np.float32)
    w = len(y) / (n_classes * np.maximum(counts, 1))
    return torch.clamp(torch.tensor(w, dtype=torch.float32), max=cap)

def local_train(in_dim, n_classes, global_state, X, y, epochs=1, lr=1e-2, mu=0.0, hidden=(64, 32),
                 class_weights=None):
    model = TwinMLP(in_dim, n_classes, hidden)
    set_state(model, global_state)
    global_ref = {k: v.clone().float() for k, v in global_state.items()}
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss(weight=class_weights)
    Xb, yb = to_tensor(X, y)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        out = model(Xb)
        loss = lossf(out, yb)
        if mu > 0:
            prox = sum((p.float() - global_ref[n]).pow(2).sum() for n, p in model.state_dict().items())
            loss = loss + (mu / 2) * prox
        loss.backward()
        opt.step()
    return get_state(model)

def evaluate(in_dim, n_classes, state, X, y, hidden=(64, 32)):
    model = TwinMLP(in_dim, n_classes, hidden)
    set_state(model, state); model.eval()
    Xb, yb = to_tensor(X, y)
    with torch.no_grad():
        out = model(Xb)
        loss = nn.CrossEntropyLoss()(out, yb).item()
        pred = out.argmax(1).numpy()
    return {'loss': loss, 'accuracy': accuracy_score(y, pred), 'f1_macro': f1_score(y, pred, average='macro', zero_division=0)}

In [ ]:
def our_gate_aggregate(candidates, weights, global_state, client_data, in_dim, n_classes,
                        veto_epsilon=0.0, veto_k=4.0, sigma_min=0.01, ablation=None, hidden=(64,32)):
    '''ablation in {None, 'absolute_only', 'mad_only'}.'''
    deltas = []
    for cd, cand in zip(client_data, candidates):
        before = evaluate(in_dim, n_classes, global_state, cd['X_shadow'], cd['y_shadow'], hidden)
        after = evaluate(in_dim, n_classes, cand, cd['X_shadow'], cd['y_shadow'], hidden)
        deltas.append(after['loss'] - before['loss'])
    deltas = np.array(deltas)
    median_delta = float(np.median(deltas))
    mad = max(float(np.median(np.abs(deltas - median_delta))), sigma_min)
    new_states, new_weights, vetoed = [], [], []
    for i in range(len(candidates)):
        is_abs = deltas[i] > veto_epsilon
        is_out = deltas[i] > median_delta + veto_k * mad
        if ablation == 'absolute_only': veto = is_abs
        elif ablation == 'mad_only': veto = is_out
        else: veto = is_abs or is_out
        if veto: vetoed.append(i)
        else: new_states.append(candidates[i]); new_weights.append(weights[i])
    new_global = average_states(new_states, new_weights) if new_states else global_state
    return new_global, deltas, vetoed


def run_federated(client_data, in_dim, n_classes, rounds=15, local_epochs=1, lr=1e-2, mu=0.0,
                   hidden=(64, 32), use_twin_gate=True, veto_epsilon=0.0, veto_k=4.0, seed=0,
                   use_class_weights=True, ablation=None):
    torch.manual_seed(seed); np.random.seed(seed)
    global_state = get_state(TwinMLP(in_dim, n_classes, hidden))
    history = {'round': [], 'n_vetoed': [], 'vetoed_clients': [], 'deltas': []}
    for r in range(1, rounds + 1):
        candidates, weights = [], []
        for cd in client_data:
            cw = compute_class_weights(cd['y_train'], n_classes) if use_class_weights else None
            candidate = local_train(in_dim, n_classes, global_state, cd['X_train'], cd['y_train'],
                                     epochs=local_epochs, lr=lr, mu=mu, hidden=hidden, class_weights=cw)
            candidates.append(candidate); weights.append(cd['X_train'].shape[0])
        if use_twin_gate:
            global_state, deltas, vetoed = our_gate_aggregate(candidates, weights, global_state, client_data,
                                                                in_dim, n_classes, veto_epsilon, veto_k,
                                                                ablation=ablation, hidden=hidden)
        else:
            global_state = average_states(candidates, weights)
            deltas, vetoed = np.zeros(len(candidates)), []
        history['round'].append(r); history['n_vetoed'].append(len(vetoed))
        history['vetoed_clients'].append(vetoed); history['deltas'].append(np.asarray(deltas).tolist())
    return global_state, history

### Baseline aggregation methods
FedAvg is `run_federated(..., use_twin_gate=False)` above. The other five are implemented here, sanity-checked (in the paper's own development) against a synthetic outlier scenario before use: robust methods stayed within ~0.1-0.5 of the honest-client consensus while plain FedAvg was pulled ~45 units away by a single malicious update.

In [ ]:
def median_aggregate(candidates, weights):
    keys = candidates[0].keys(); out = {}
    for k in keys:
        stacked = torch.stack([c[k].float() for c in candidates], dim=0)
        out[k] = stacked.median(dim=0).values
    return out

def trimmed_mean_aggregate(candidates, weights, beta=0.2):
    keys = candidates[0].keys(); n = len(candidates)
    k_trim = max(0, int(np.floor(beta * n))); out = {}
    for key in keys:
        stacked = torch.stack([c[key].float() for c in candidates], dim=0)
        sorted_vals, _ = torch.sort(stacked, dim=0)
        trimmed = sorted_vals[k_trim:n-k_trim] if (2*k_trim < n and k_trim > 0) else sorted_vals
        out[key] = trimmed.mean(dim=0)
    return out

def krum_aggregate(candidates, weights, f=1, multi=False, m=None):
    n = len(candidates)
    vecs = [state_to_vec(c) for c in candidates]
    dists = torch.zeros(n, n)
    for i in range(n):
        for j in range(n):
            if i != j: dists[i, j] = torch.sum((vecs[i]-vecs[j])**2)
    n_neighbors = max(1, n - f - 2)
    scores = []
    for i in range(n):
        d = dists[i].clone(); d[i] = float('inf')
        nearest = torch.topk(d, k=min(n_neighbors, n-1), largest=False).values
        scores.append(nearest.sum().item())
    order = np.argsort(scores)
    if multi:
        m = m or max(1, n - f)
        chosen = order[:m]
        return average_states([candidates[i] for i in chosen], [weights[i] for i in chosen])
    return candidates[order[0]]

def fltrust_aggregate(candidates, weights, global_state, root_state_delta):
    ref_vec = root_state_delta; ref_norm = torch.norm(ref_vec) + 1e-8
    trust_scores, client_deltas = [], []
    for c in candidates:
        c_vec = state_to_vec(c) - state_to_vec(global_state)
        c_norm = torch.norm(c_vec) + 1e-8
        cos_sim = torch.dot(c_vec, ref_vec) / (c_norm * ref_norm)
        ts = max(0.0, cos_sim.item())
        trust_scores.append(ts)
        client_deltas.append(c_vec * (ref_norm / c_norm))
    total_ts = sum(trust_scores)
    if total_ts <= 1e-8: return global_state
    agg_delta = sum(ts*d for ts, d in zip(trust_scores, client_deltas)) / total_ts
    new_vec = state_to_vec(global_state) + agg_delta
    return vec_to_state_like(new_vec, global_state)


def run_federated_method(method, client_data, in_dim, n_classes, rounds=15, lr=5e-3, seed=0,
                          f_assumed=1, root_X=None, root_y=None, attacker_idx=None):
    '''Unified runner for all 7 methods (fedavg, median, trimmed_mean, krum, multi_krum,
    fltrust, our_gate) -- one shared local_train/evaluate path underneath every one,
    so cross-method comparisons are apples-to-apples.'''
    torch.manual_seed(seed); np.random.seed(seed)
    global_state = get_state(TwinMLP(in_dim, n_classes))
    caught = 0
    for r in range(rounds):
        candidates, weights = [], []
        for cd in client_data:
            cw = compute_class_weights(cd['y_train'], n_classes)
            cand = local_train(in_dim, n_classes, global_state, cd['X_train'], cd['y_train'],
                                lr=lr, class_weights=cw)
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        if method == 'fedavg':
            global_state = average_states(candidates, weights)
        elif method == 'median':
            global_state = median_aggregate(candidates, weights)
        elif method == 'trimmed_mean':
            global_state = trimmed_mean_aggregate(candidates, weights, beta=0.2)
        elif method == 'krum':
            global_state = krum_aggregate(candidates, weights, f=f_assumed, multi=False)
        elif method == 'multi_krum':
            global_state = krum_aggregate(candidates, weights, f=f_assumed, multi=True, m=len(candidates)-f_assumed)
        elif method == 'fltrust':
            root_cw = compute_class_weights(root_y, n_classes)
            root_cand = local_train(in_dim, n_classes, global_state, root_X, root_y, lr=lr, class_weights=root_cw)
            root_delta = state_to_vec(root_cand) - state_to_vec(global_state)
            global_state = fltrust_aggregate(candidates, weights, global_state, root_delta)
        elif method == 'our_gate':
            global_state, deltas, vetoed = our_gate_aggregate(candidates, weights, global_state, client_data,
                                                                in_dim, n_classes, VETO_EPSILON, VETO_K)
            if attacker_idx is not None and attacker_idx in vetoed: caught += 1
        else:
            raise ValueError(method)
    return global_state, caught

### Attack implementations
**Blunt**: full random relabeling of a compromised twin's local labels. **Stealthy**: relabels only the fraction of a source class's samples closest (by margin) to the most naturally confusable other class -- a realistic 'stay under the radar' attacker. **Adaptive**: uses the compromised twin's own shadow split (which it genuinely has access to) to pick the highest poisoning severity that doesn't look like a regression by its own signal -- cannot see other twins' deltas, so cannot evade the peer-relative test directly.

In [ ]:
def poison_blunt(y, severity, n_classes, seed):
    rng = np.random.default_rng(seed)
    mask = rng.random(len(y)) < severity
    y_new = y.copy(); y_new[mask] = rng.integers(0, n_classes, size=mask.sum())
    return y_new

def most_confusable_pair(class_means):
    n = len(class_means); best = (0, 1, np.inf)
    for i in range(n):
        for j in range(i+1, n):
            d = np.linalg.norm(class_means[i] - class_means[j])
            if d < best[2]: best = (i, j, d)
    return best[0], best[1]

def poison_stealthy(X_train, y_train, class_means, source_cls, target_cls, frac, seed):
    y_new = y_train.copy()
    src_idx = np.where(y_train == source_cls)[0]
    if len(src_idx) == 0: return y_new
    own_dist = np.linalg.norm(X_train[src_idx] - class_means[source_cls], axis=1)
    tgt_dist = np.linalg.norm(X_train[src_idx] - class_means[target_cls], axis=1)
    order = src_idx[np.argsort(tgt_dist - own_dist)]
    y_new[order[:int(len(order) * frac)]] = target_cls
    return y_new

def adaptive_attacker_choose_severity(gstate, cd, in_dim, n_classes, lr, candidate_severities, round_seed):
    best_severity, best_state = candidate_severities[0], None
    before = evaluate(in_dim, n_classes, gstate, cd['X_shadow'], cd['y_shadow'])
    for s in candidate_severities:
        y_poisoned = poison_blunt(cd['y_train'], s, n_classes, seed=round_seed)
        cw = compute_class_weights(y_poisoned, n_classes)
        cand = local_train(in_dim, n_classes, gstate, cd['X_train'], y_poisoned, lr=lr, class_weights=cw)
        after = evaluate(in_dim, n_classes, cand, cd['X_shadow'], cd['y_shadow'])
        if after['loss'] - before['loss'] <= 0:
            best_severity, best_state = s, cand
    if best_state is None:
        y_poisoned = poison_blunt(cd['y_train'], candidate_severities[0], n_classes, seed=round_seed)
        cw = compute_class_weights(y_poisoned, n_classes)
        best_state = local_train(in_dim, n_classes, gstate, cd['X_train'], y_poisoned, lr=lr, class_weights=cw)
    return best_state, best_severity

## 6-15. Experiment functions
Each function below is self-contained: it takes `dataset` (and sometimes pre-loaded data) as an argument, runs its experiment, saves its own result file(s) under `results/{dataset}/`, and returns a small summary dict. None of them print-and-hope -- every number that matters is written to disk before the function returns. The orchestration loop in Section 16 calls these, wrapped in try/except, for all three datasets.

### 6. Single-attacker validation -> `table4_single_attacker.csv`, `confusion_matrix.png`, `classification_report.txt`

In [ ]:
def exp_single_attacker(dataset, outdir, seed=1):
    clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, MAX_ROWS_FULL)
    in_dim = len(feats)
    poison_clients = copy.deepcopy(clients)
    rng = np.random.default_rng(seed)
    poison_clients[POISON_CLIENT]['y_train'] = rng.integers(0, nc, size=len(poison_clients[POISON_CLIENT]['y_train']))

    Xall = np.concatenate([c['X_train'] for c in clients] + [c['X_shadow'] for c in clients])
    yall = np.concatenate([c['y_train'] for c in clients] + [c['y_shadow'] for c in clients])
    cstate = local_train(in_dim, nc, get_state(TwinMLP(in_dim, nc)), Xall, yall,
                          epochs=LOCAL_EPOCHS*ROUNDS, lr=LR, class_weights=compute_class_weights(yall, nc))
    results = {'centralized': evaluate(in_dim, nc, cstate, Xte, yte)}
    gstates = {}
    for cond, cl, gate in [('clean_gate', clients, True), ('clean_nogate', clients, False),
                           ('poison_gate', poison_clients, True), ('poison_nogate', poison_clients, False)]:
        g, h = run_federated(cl, in_dim, nc, rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, lr=LR,
                              mu=FEDPROX_MU, use_twin_gate=gate, veto_epsilon=VETO_EPSILON, veto_k=VETO_K, seed=seed)
        results[cond] = evaluate(in_dim, nc, g, Xte, yte)
        gstates[cond] = g
        if cond == 'clean_gate': h_clean_gate = h
        if cond == 'poison_gate': h_poison_gate = h

    rows = [{'condition': k, **v} for k, v in results.items()]
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table4_single_attacker.csv', index=False)
    fp_rate = sum(h_clean_gate['n_vetoed'])
    with open(f'{outdir}/table4_veto_trace.json', 'w') as f:
        json.dump({'vetoed_attacked': h_poison_gate['vetoed_clients'], 'vetoed_clean': h_clean_gate['vetoed_clients'],
                   'clean_fp_rate': f'{fp_rate}/{ROUNDS*len(clients)}'}, f, indent=2)

    model = TwinMLP(in_dim, nc); set_state(model, gstates['poison_gate']); model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(Xte, dtype=torch.float32)).argmax(1).numpy()
    with open(f'{outdir}/table4_classification_report.txt', 'w') as f:
        f.write(classification_report(yte, pred, target_names=le.classes_, zero_division=0))
    cm = confusion_matrix(yte, pred)
    # Save raw counts as data, not only as a picture -- so a normalized or re-styled figure
    # can be rebuilt later without re-running the whole federated training pipeline.
    pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_csv(f'{outdir}/table4_confusion_matrix_counts.csv')
    # Normalize by ROW (true class) before plotting. On severely imbalanced data, a raw-count
    # color scale is dominated by the majority class and makes minority-class performance
    # invisible even when it is genuinely good (e.g. 99% recall on a rare class can look
    # identical to 0% if that class's absolute count is small) -- caught by inspecting an
    # earlier version of this figure against its own classification report and finding they
    # visually contradicted each other.
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(nc)); ax.set_xticklabels(le.classes_, rotation=45, ha='right')
    ax.set_yticks(range(nc)); ax.set_yticklabels(le.classes_)
    for i in range(nc):
        for j in range(nc):
            if cm[i, j] > 0:
                color = 'white' if cm_norm[i, j] > 0.5 else 'black'
                ax.text(j, i, f'{cm[i,j]}', ha='center', va='center', fontsize=6, color=color)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True (row-normalized color)')
    ax.set_title(f'Confusion matrix -- {dataset}, real data, gated, attacked')
    plt.colorbar(im, label='Recall (fraction of true class)'); plt.tight_layout()
    plt.savefig(f'{outdir}/confusion_matrix.png', dpi=150); plt.close(fig)
    return {'section': 'single_attacker', 'dataset': dataset, 'summary': results}

### 7. Six-baseline comparison -> `table5_baselines.csv`

In [ ]:
def exp_baseline_comparison(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    methods = ['fedavg', 'median', 'trimmed_mean', 'krum', 'multi_krum', 'fltrust', 'our_gate']
    rows = []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        n_poison = max(1, len(clients) // 3)
        poison = copy.deepcopy(clients)
        rng = np.random.default_rng(seed)
        for pc in range(n_poison):
            poison[pc]['y_train'] = rng.integers(0, nc, size=len(poison[pc]['y_train']))
        Xall = np.concatenate([c['X_train'] for c in clients]); yall = np.concatenate([c['y_train'] for c in clients])
        root_idx = rng.choice(len(Xall), size=min(500, len(Xall)), replace=False)
        root_X, root_y = Xall[root_idx], yall[root_idx]
        for method in methods:
            g_clean, _ = run_federated_method(method, clients, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed,
                                               f_assumed=n_poison, root_X=root_X, root_y=root_y)
            r_clean = evaluate(len(feats), nc, g_clean, Xte, yte)
            g_poison, _ = run_federated_method(method, poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed,
                                                f_assumed=n_poison, root_X=root_X, root_y=root_y)
            r_poison = evaluate(len(feats), nc, g_poison, Xte, yte)
            rows.append(dict(dataset=dataset, seed=seed, method=method,
                              clean_f1=r_clean['f1_macro'], poison_f1=r_poison['f1_macro'],
                              clean_acc=r_clean['accuracy'], poison_acc=r_poison['accuracy']))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table5_baselines.csv', index=False)
    return {'section': 'baseline_comparison', 'dataset': dataset,
            'summary': rdf.groupby('method')[['clean_f1','poison_f1']].mean().to_dict()}

### 8. Veto-condition ablation: blunt + stealthy -> `table6_ablation_blunt.csv`, `table7_ablation_stealthy.csv`

In [ ]:
def run_gate_variant(clients, in_dim_, nc_, ablation, seed, attacker_idx=0):
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    caught = 0
    for r in range(ROUNDS):
        candidates, weights = [], []
        for cd in clients:
            cw = compute_class_weights(cd['y_train'], nc_)
            cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed = our_gate_aggregate(candidates, weights, gstate, clients, in_dim_, nc_,
                                                     VETO_EPSILON, VETO_K, ablation=ablation)
        if attacker_idx in vetoed: caught += 1
    return caught


def exp_ablation(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    blunt_rows, stealthy_rows = [], []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        n_poison = max(1, len(clients) // 3)
        Xall = np.concatenate([c['X_train'] for c in clients]); yall = np.concatenate([c['y_train'] for c in clients])
        class_means = np.array([Xall[yall == c].mean(axis=0) for c in range(nc)])
        src, tgt = most_confusable_pair(class_means)
        blunt = copy.deepcopy(clients)
        for pc in range(n_poison):
            blunt[pc]['y_train'] = poison_blunt(blunt[pc]['y_train'], 1.0, nc, seed)
        stealthy = copy.deepcopy(clients)
        stealthy[0]['y_train'] = poison_stealthy(stealthy[0]['X_train'], stealthy[0]['y_train'],
                                                   class_means, src, tgt, frac=0.9, seed=seed)
        for ablation_name in ['absolute_only', 'mad_only', None]:
            label = ablation_name or 'combined'
            caught_b = run_gate_variant(blunt, len(feats), nc, ablation_name, seed, attacker_idx=0)
            blunt_rows.append(dict(dataset=dataset, variant=label, seed=seed, caught=caught_b))
            caught_s = run_gate_variant(stealthy, len(feats), nc, ablation_name, seed, attacker_idx=0)
            stealthy_rows.append(dict(dataset=dataset, variant=label, seed=seed, caught=caught_s))
    pd.DataFrame(blunt_rows).to_csv(f'{outdir}/table6_ablation_blunt.csv', index=False)
    pd.DataFrame(stealthy_rows).to_csv(f'{outdir}/table7_ablation_stealthy.csv', index=False)
    return {'section': 'ablation', 'dataset': dataset,
            'summary': {'blunt': pd.DataFrame(blunt_rows).groupby('variant')['caught'].mean().to_dict(),
                        'stealthy': pd.DataFrame(stealthy_rows).groupby('variant')['caught'].mean().to_dict()}}

### 9. Locally-adaptive attacker -> `table8_adaptive.csv`

In [ ]:
def run_federated_adaptive(clients, attacker_idx, in_dim_, nc_, seed, adaptive=True, fixed_severity=1.0):
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    caught = 0
    for r in range(ROUNDS):
        candidates, weights = [], []
        for i, cd in enumerate(clients):
            if i == attacker_idx:
                if adaptive:
                    cand, sev = adaptive_attacker_choose_severity(gstate, cd, in_dim_, nc_, LR,
                        [0.1,0.3,0.5,0.7,0.9,1.0], round_seed=seed*100+r)
                else:
                    y_p = poison_blunt(cd['y_train'], fixed_severity, nc_, seed=seed*100+r)
                    cw = compute_class_weights(y_p, nc_)
                    cand = local_train(in_dim_, nc_, gstate, cd['X_train'], y_p, lr=LR, class_weights=cw)
            else:
                cw = compute_class_weights(cd['y_train'], nc_)
                cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed = our_gate_aggregate(candidates, weights, gstate, clients, in_dim_, nc_,
                                                     VETO_EPSILON, VETO_K)
        if attacker_idx in vetoed: caught += 1
    return gstate, caught


def exp_adaptive_attacker(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    rows = []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        g_fixed, caught_fixed = run_federated_adaptive(clients, 0, len(feats), nc, seed, adaptive=False)
        r_fixed = evaluate(len(feats), nc, g_fixed, Xte, yte)
        g_adapt, caught_adapt = run_federated_adaptive(clients, 0, len(feats), nc, seed, adaptive=True)
        r_adapt = evaluate(len(feats), nc, g_adapt, Xte, yte)
        rows.append(dict(dataset=dataset, seed=seed, fixed_caught=caught_fixed, fixed_f1=r_fixed['f1_macro'],
                          adaptive_caught=caught_adapt, adaptive_f1=r_adapt['f1_macro']))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table8_adaptive.csv', index=False)
    return {'section': 'adaptive_attacker', 'dataset': dataset, 'summary': rdf.mean(numeric_only=True).to_dict()}

### 10. Threshold sensitivity: k and W_max -> `table9_sensitivity_k.csv`, `table9_sensitivity_wmax.csv`

In [ ]:
def exp_sensitivity(dataset, outdir):
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    sens_seeds = [100] if QUICK_MODE else [100, 200]
    k_rows, w_rows = [], []
    for seed in sens_seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        n_poison = max(1, len(clients) // 3)
        poison = copy.deepcopy(clients)
        rng = np.random.default_rng(seed)
        for pc in range(n_poison):
            poison[pc]['y_train'] = rng.integers(0, nc, size=len(poison[pc]['y_train']))
        for k in [2.0, 3.0, 4.0, 5.0, 6.0]:
            g, h = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed,
                                  use_twin_gate=True, veto_epsilon=VETO_EPSILON, veto_k=k)
            r = evaluate(len(feats), nc, g, Xte, yte)
            k_rows.append(dict(dataset=dataset, seed=seed, k=k, poison_f1=r['f1_macro']))
        global WEIGHT_CAP
        old_cap = WEIGHT_CAP
        for wcap in [5.0, 10.0, 20.0, 30.0, 50.0]:
            WEIGHT_CAP = wcap
            g, h = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed,
                                  use_twin_gate=True, veto_epsilon=VETO_EPSILON, veto_k=VETO_K)
            r = evaluate(len(feats), nc, g, Xte, yte)
            w_rows.append(dict(dataset=dataset, seed=seed, wcap=wcap, poison_f1=r['f1_macro']))
        WEIGHT_CAP = old_cap
    pd.DataFrame(k_rows).to_csv(f'{outdir}/table9_sensitivity_k.csv', index=False)
    pd.DataFrame(w_rows).to_csv(f'{outdir}/table9_sensitivity_wmax.csv', index=False)
    return {'section': 'sensitivity', 'dataset': dataset,
            'summary': {'k': pd.DataFrame(k_rows).groupby('k')['poison_f1'].mean().to_dict(),
                        'wmax': pd.DataFrame(w_rows).groupby('wcap')['poison_f1'].mean().to_dict()}}

### 11. Long-horizon (50-round) drift + epsilon=0.05 fix -> `longhorizon.csv`

In [ ]:
def run_federated_long(clients, in_dim_, nc_, rounds, seed, veto_epsilon=0.0):
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    cum_vetoed = 0; trace = []
    for r in range(1, rounds+1):
        candidates, weights = [], []
        for cd in clients:
            cw = compute_class_weights(cd['y_train'], nc_)
            cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed = our_gate_aggregate(candidates, weights, gstate, clients, in_dim_, nc_,
                                                     veto_epsilon, VETO_K)
        cum_vetoed += len(vetoed)
        if r in (15, 30, 50): trace.append((r, cum_vetoed))
    return cum_vetoed, trace


def exp_long_horizon(dataset, outdir, seed=1):
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
    _, trace_e0 = run_federated_long(clients, len(feats), nc, 50, seed, veto_epsilon=0.0)
    _, trace_e05 = run_federated_long(clients, len(feats), nc, 50, seed, veto_epsilon=0.05)
    n_poison = max(1, len(clients) // 3)
    poison = copy.deepcopy(clients)
    rng = np.random.default_rng(seed)
    for pc in range(n_poison):
        poison[pc]['y_train'] = rng.integers(0, nc, size=len(poison[pc]['y_train']))
    detection_rows = []
    for eps in [0.0, 0.05]:
        g, h = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed,
                              use_twin_gate=True, veto_epsilon=eps, veto_k=VETO_K)
        r = evaluate(len(feats), nc, g, Xte, yte)
        caught = sum(1 for v in h['vetoed_clients'] if any(p < n_poison for p in v))
        detection_rows.append(dict(dataset=dataset, eps=eps, attacked_f1=r['f1_macro'], caught_rounds=caught))
    out = {'dataset': dataset, 'seed': seed,
           'drift_trace_eps0': trace_e0, 'drift_trace_eps005': trace_e05,
           'detection_at_15_rounds': detection_rows}
    with open(f'{outdir}/longhorizon.json', 'w') as f:
        json.dump(out, f, indent=2)
    return {'section': 'long_horizon', 'dataset': dataset, 'summary': out}

### 12. Cross-dataset generalization row -> `table10_generalization.csv`

In [ ]:
def exp_generalization(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    rows = []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        poison = copy.deepcopy(clients)
        rng = np.random.default_rng(seed)
        poison[0]['y_train'] = rng.integers(0, nc, size=len(poison[0]['y_train']))
        g_cg, h_cg = run_federated(clients, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=True)
        g_cn, h_cn = run_federated(clients, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=False)
        g_pg, h_pg = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=True)
        g_pn, h_pn = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=False)
        acc_cg = evaluate(len(feats), nc, g_cg, Xte, yte)['accuracy']
        acc_cn = evaluate(len(feats), nc, g_cn, Xte, yte)['accuracy']
        acc_pg = evaluate(len(feats), nc, g_pg, Xte, yte)['accuracy']
        acc_pn = evaluate(len(feats), nc, g_pn, Xte, yte)['accuracy']
        fp = sum(h_cg['n_vetoed']); n_clean_rounds = ROUNDS * len(clients)
        caught = sum(1 for v in h_pg['vetoed_clients'] if 0 in v)
        rows.append(dict(dataset=dataset, seed=seed, clean_gate=acc_cg, clean_nogate=acc_cn,
                          poison_gate=acc_pg, poison_nogate=acc_pn, fp_rate=fp/n_clean_rounds,
                          catch_rate=caught/ROUNDS, gate_recovers=acc_pg-acc_pn))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table10_generalization.csv', index=False)
    return {'section': 'generalization', 'dataset': dataset, 'summary': rdf.mean(numeric_only=True).to_dict()}

### 13. Real severity sweep -- the headline result -> `table_headline_severity_sweep.csv`, `headline_plot.png`

In [ ]:
def exp_severity_sweep(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    rows = []
    for n_poisoned in [1, 2, 3]:
        for seed in seeds:
            clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, MAX_ROWS_FULL)
            poison = copy.deepcopy(clients)
            rng = np.random.default_rng(seed)
            for pc in range(n_poisoned):
                poison[pc]['y_train'] = rng.integers(0, nc, size=len(poison[pc]['y_train']))
            g_gate, h_gate = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=True)
            g_nogate, h_nogate = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=False)
            r_gate = evaluate(len(feats), nc, g_gate, Xte, yte)
            r_nogate = evaluate(len(feats), nc, g_nogate, Xte, yte)
            caught = sum(1 for v in h_gate['vetoed_clients'] if any(p < n_poisoned for p in v))
            rows.append(dict(dataset=dataset, n_poisoned=n_poisoned, seed=seed, acc_gate=r_gate['accuracy'],
                              acc_nogate=r_nogate['accuracy'], f1_gate=r_gate['f1_macro'],
                              f1_nogate=r_nogate['f1_macro'], caught_rounds=caught))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table_headline_severity_sweep.csv', index=False)
    summary = rdf.groupby('n_poisoned')[['acc_nogate','acc_gate','f1_nogate','f1_gate','caught_rounds']].mean()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(summary.index, summary['acc_nogate'], 'o-', label='No gate', color='#C44E52')
    axes[0].plot(summary.index, summary['acc_gate'], 's-', label='With gate', color='#55A868')
    axes[0].set_title('Accuracy vs. severity'); axes[0].legend()
    axes[1].plot(summary.index, summary['f1_nogate'], 'o-', label='No gate', color='#C44E52')
    axes[1].plot(summary.index, summary['f1_gate'], 's-', label='With gate', color='#55A868')
    axes[1].set_title('Macro-F1 vs. severity'); axes[1].legend()
    plt.suptitle(f'{dataset}, real data')
    plt.tight_layout()
    plt.savefig(f'{outdir}/headline_plot.png', dpi=150); plt.close(fig)
    return {'section': 'severity_sweep', 'dataset': dataset, 'summary': summary.to_dict()}

### 14. Explainability (SHAP) -> `shap_importance.png`, `shap_values.csv`

In [ ]:
def exp_shap(dataset, outdir, seed=1):
    clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, MAX_ROWS_FULL)
    in_dim = len(feats)
    g_gate, h = run_federated(clients, in_dim, nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=True)
    model = TwinMLP(in_dim, nc); set_state(model, g_gate); model.eval()
    background = torch.tensor(Xte[:150], dtype=torch.float32)
    idx = np.random.default_rng(0).choice(len(Xte), size=min(200, len(Xte)), replace=False)
    sample = torch.tensor(Xte[idx], dtype=torch.float32)
    explainer = shap.DeepExplainer(model, background)
    sv = np.array(explainer.shap_values(sample))
    mean_abs = np.abs(sv).mean(axis=(0, 2))
    order_feat = np.argsort(mean_abs)[::-1]
    pd.DataFrame({'feature': [feats[i] for i in order_feat], 'mean_abs_shap': mean_abs[order_feat]}).to_csv(
        f'{outdir}/shap_values.csv', index=False)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh([feats[i] for i in order_feat[:15]][::-1], mean_abs[order_feat[:15]][::-1], color='#4C72B0')
    ax.set_xlabel('mean |SHAP value|'); ax.set_title(f'Feature importance -- {dataset}, real data')
    plt.tight_layout()
    plt.savefig(f'{outdir}/shap_importance.png', dpi=150); plt.close(fig)
    return {'section': 'shap', 'dataset': dataset, 'summary': {'top5': [feats[i] for i in order_feat[:5]]}}

### 15. False-veto fairness -> `fairness.csv`

In [ ]:
def run_gate_track_vetoes(clients, in_dim_, nc_, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    veto_counts = np.zeros(len(clients), dtype=int)
    for r in range(ROUNDS):
        candidates, weights = [], []
        for cd in clients:
            cw = compute_class_weights(cd['y_train'], nc_)
            cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed = our_gate_aggregate(candidates, weights, gstate, clients, in_dim_, nc_,
                                                     VETO_EPSILON, VETO_K)
        for i in vetoed: veto_counts[i] += 1
    return veto_counts


def exp_fairness(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    rows = []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        vc = run_gate_track_vetoes(clients, len(feats), nc, seed)
        for i, c in enumerate(clients):
            dom_frac = np.bincount(c['y_train']).max() / len(c['y_train'])
            rows.append(dict(dataset=dataset, seed=seed, client=i, n_train=len(c['y_train']),
                              dominant_frac=dom_frac, vetoes=vc[i]))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/fairness.csv', index=False)
    corr = rdf[['dominant_frac','vetoes']].corr().iloc[0,1]
    return {'section': 'fairness', 'dataset': dataset,
            'summary': {'total_false_vetoes': int(rdf['vetoes'].sum()), 'corr_dominant_vetoes': float(corr) if pd.notna(corr) else None}}

## 16. Stronger Byzantine attacks: scaling, sign-flipping, model-replacement/backdoor

The label-flipping attacks above (blunt, stealthy) are one threat family. These three are qualitatively different: they craft the **parameter update itself** rather than corrupting training labels, following the standard Byzantine-FL and model-replacement (Bagdasaryan et al. 2020) literature. Each is tested against the gate the same way: does the veto mechanism catch it, and what does it do to the global model if it doesn't.

In [ ]:
def poison_scaling(local_state, global_state, lam=10.0):
    '''theta_i' = theta_G + lambda*(theta_i - theta_G). Amplifies an honestly-trained
    update's distance from the global state by a large factor -- tries to dominate
    the aggregate by sheer magnitude rather than by corrupting what was learned.'''
    scaled = {}
    for k in local_state:
        scaled[k] = global_state[k].float() + lam * (local_state[k].float() - global_state[k].float())
    return scaled

def poison_sign_flip(local_state, global_state, lam=1.0):
    '''delta_theta_i' = -lambda * delta_theta_i. Flips the sign of the honestly-computed
    update direction -- actively pushes the global model AWAY from what local training
    learned, rather than merely corrupting the training signal.'''
    flipped = {}
    for k in local_state:
        delta = local_state[k].float() - global_state[k].float()
        flipped[k] = global_state[k].float() - lam * delta
    return flipped

def poison_model_replacement(in_dim, n_classes, global_state, X, y, lr, target_class=0, boost=None, n_twins=8):
    '''Model-replacement / backdoor (Bagdasaryan et al. 2020): train toward an attacker-
    chosen target label for ALL local data (the backdoor objective), then boost-scale the
    resulting delta by ~n_twins so that after federated averaging divides it back down,
    the attacker's chosen behavior survives largely intact rather than being diluted.'''
    y_backdoor = np.full_like(y, target_class)
    cw = compute_class_weights(y_backdoor, n_classes)
    cand = local_train(in_dim, n_classes, global_state, X, y_backdoor, lr=lr, class_weights=cw)
    boost = boost if boost is not None else n_twins
    boosted = {}
    for k in cand:
        delta = cand[k].float() - global_state[k].float()
        boosted[k] = global_state[k].float() + boost * delta
    return boosted

In [ ]:
def run_federated_strong_attack(clients, attacker_idx, in_dim_, nc_, attack_type, seed, rounds=None,
                                 lam=10.0, boost=None):
    '''attack_type in {'scaling', 'sign_flip', 'model_replacement'}. Runs the standard gate
    loop but the compromised twin's submitted update is crafted directly rather than by
    corrupting its local labels before training (except model_replacement, which does both).'''
    rounds = rounds or ROUNDS
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    caught = 0
    for r in range(rounds):
        candidates, weights = [], []
        for i, cd in enumerate(clients):
            cw = compute_class_weights(cd['y_train'], nc_)
            honest_cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            if i == attacker_idx:
                if attack_type == 'scaling':
                    cand = poison_scaling(honest_cand, gstate, lam=lam)
                elif attack_type == 'sign_flip':
                    cand = poison_sign_flip(honest_cand, gstate, lam=lam)
                elif attack_type == 'model_replacement':
                    cand = poison_model_replacement(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'],
                                                     LR, target_class=0, boost=boost, n_twins=len(clients))
                else:
                    raise ValueError(attack_type)
            else:
                cand = honest_cand
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed = our_gate_aggregate(candidates, weights, gstate, clients, in_dim_, nc_,
                                                     VETO_EPSILON, VETO_K)
        if attacker_idx in vetoed: caught += 1
    return gstate, caught


def exp_strong_attacks(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    rows = []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        for attack_type, lam in [('scaling', 10.0), ('sign_flip', 1.0), ('model_replacement', None)]:
            g, caught = run_federated_strong_attack(clients, 0, len(feats), nc, attack_type, seed, lam=lam or 10.0)
            r = evaluate(len(feats), nc, g, Xte, yte)
            rows.append(dict(dataset=dataset, attack=attack_type, seed=seed,
                              caught=caught, f1=r['f1_macro'], acc=r['accuracy']))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table_strong_attacks.csv', index=False)
    return {'section': 'strong_attacks', 'dataset': dataset,
            'summary': rdf.groupby('attack')[['caught','f1']].mean().to_dict()}

## 17. A peer-aware adaptive attacker (Kerckhoffs-style)

The adaptive attacker in the main pipeline only sees its own shadow loss and cannot evade the peer-relative MAD test directly. This one is strictly stronger: it is given $\varepsilon$, $k$, and an estimate of the **peer threshold** for the current round (either the true current-round median/MAD -- an oracle, worst-case bound on what a fully-informed attacker could achieve -- or the *previous* round's median/MAD, a more realistic capability for an attacker that has passively observed several rounds of global-model updates). It then searches for the highest severity that stays under **both** its own absolute signal and the peer threshold.

In [ ]:
def adaptive_attacker_peer_aware(gstate, cd, in_dim_, nc_, lr, candidate_severities, round_seed,
                                   peer_threshold=None):
    '''peer_threshold: median(delta)+k*MAD(delta) for the round, supplied by the harness
    (oracle: true current-round value; realistic: previous round's value). None = attacker
    only has its own absolute signal (equivalent to the non-peer-aware adaptive attacker).'''
    best_severity, best_state = candidate_severities[0], None
    before = evaluate(in_dim_, nc_, gstate, cd['X_shadow'], cd['y_shadow'])
    effective_cap = min(VETO_EPSILON, peer_threshold) if peer_threshold is not None else VETO_EPSILON
    for s in candidate_severities:
        y_poisoned = poison_blunt(cd['y_train'], s, nc_, seed=round_seed)
        cw = compute_class_weights(y_poisoned, nc_)
        cand = local_train(in_dim_, nc_, gstate, cd['X_train'], y_poisoned, lr=lr, class_weights=cw)
        after = evaluate(in_dim_, nc_, cand, cd['X_shadow'], cd['y_shadow'])
        if after['loss'] - before['loss'] <= effective_cap:
            best_severity, best_state = s, cand
    if best_state is None:
        y_poisoned = poison_blunt(cd['y_train'], candidate_severities[0], nc_, seed=round_seed)
        cw = compute_class_weights(y_poisoned, nc_)
        best_state = local_train(in_dim_, nc_, gstate, cd['X_train'], y_poisoned, lr=lr, class_weights=cw)
    return best_state, best_severity


def run_federated_peer_aware(clients, attacker_idx, in_dim_, nc_, seed, mode='oracle', rounds=None):
    '''mode: 'oracle' (attacker sees the TRUE current-round peer threshold -- worst-case upper
    bound on attacker capability) or 'realistic' (attacker uses the PREVIOUS round's threshold,
    a passively-observable quantity in a real deployment).'''
    rounds = rounds or ROUNDS
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    caught = 0
    prev_threshold = None
    for r in range(rounds):
        # In oracle mode we need honest candidates FIRST to compute the true round threshold,
        # then let the attacker react to it -- an explicit, disclosed simulation-only privilege.
        honest_candidates, honest_weights = [], []
        for i, cd in enumerate(clients):
            if i == attacker_idx: continue
            cw = compute_class_weights(cd['y_train'], nc_)
            cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            honest_candidates.append((i, cand)); honest_weights.append(cd['X_train'].shape[0])
        honest_deltas = []
        for i, cand in honest_candidates:
            before = evaluate(in_dim_, nc_, gstate, clients[i]['X_shadow'], clients[i]['y_shadow'])
            after = evaluate(in_dim_, nc_, cand, clients[i]['X_shadow'], clients[i]['y_shadow'])
            honest_deltas.append(after['loss'] - before['loss'])
        honest_deltas = np.array(honest_deltas)
        med = float(np.median(honest_deltas)); mad = max(float(np.median(np.abs(honest_deltas - med))), 0.01)
        true_threshold = med + VETO_K * mad
        peer_threshold = true_threshold if mode == 'oracle' else prev_threshold
        atk_state, sev = adaptive_attacker_peer_aware(gstate, clients[attacker_idx], in_dim_, nc_, LR,
            [0.1,0.3,0.5,0.7,0.9,1.0], round_seed=seed*100+r, peer_threshold=peer_threshold)
        candidates, weights = [None]*len(clients), [0]*len(clients)
        for i, cand in honest_candidates: candidates[i] = cand; weights[i] = clients[i]['X_train'].shape[0]
        candidates[attacker_idx] = atk_state; weights[attacker_idx] = clients[attacker_idx]['X_train'].shape[0]
        gstate, deltas, vetoed = our_gate_aggregate(candidates, weights, gstate, clients, in_dim_, nc_,
                                                     VETO_EPSILON, VETO_K)
        if attacker_idx in vetoed: caught += 1
        prev_threshold = true_threshold
    return gstate, caught


def exp_peer_aware_attacker(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    rows = []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        for mode in ['realistic', 'oracle']:
            g, caught = run_federated_peer_aware(clients, 0, len(feats), nc, seed, mode=mode)
            r = evaluate(len(feats), nc, g, Xte, yte)
            rows.append(dict(dataset=dataset, mode=mode, seed=seed, caught=caught, f1=r['f1_macro']))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table_peer_aware_attacker.csv', index=False)
    return {'section': 'peer_aware_attacker', 'dataset': dataset,
            'summary': rdf.groupby('mode')[['caught','f1']].mean().to_dict()}

## 18. Controlled non-IID severity sweep (Dirichlet partition)

The device-based partition used everywhere else in this notebook is non-IID by construction, but its *severity* is whatever the real device/pseudo-device structure happens to produce -- not a controlled variable. This sweep replaces device-based partitioning with a Dirichlet($\alpha$) label-skew partition, $\alpha \in \{0.01, 0.1, 0.5, 1.0, 10.0\}$ (extreme non-IID to near-IID), and asks the specific question a peer-relative detector needs to answer: **does the gate mistake an honest-but-unusually-distributed client for an attacker?** This is measured with NO poisoning present at all -- any veto here is, by construction, a false positive against legitimate heterogeneity, not a caught attack.

In [ ]:
def make_clients_dirichlet(df, feature_names, label_col, n_clients, alpha, shadow_frac=0.25, seed=0):
    '''Standard Dirichlet label-skew partition (Hsu et al. 2019 / widely used FL non-IID
    benchmark): each client's class proportions are drawn from Dir(alpha * ones(n_classes)).
    Small alpha -> a client's data is dominated by 1-2 classes (severe skew); large alpha ->
    proportions approach uniform (near-IID). This is INDEPENDENT of device identity.'''
    rng = np.random.default_rng(seed)
    y_all = df[label_col].values
    n_classes = int(y_all.max()) + 1
    idx_by_class = [np.where(y_all == c)[0] for c in range(n_classes)]
    for idx in idx_by_class: rng.shuffle(idx)
    client_indices = [[] for _ in range(n_clients)]
    for c in range(n_classes):
        if len(idx_by_class[c]) == 0: continue
        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        splits = (np.cumsum(proportions) * len(idx_by_class[c])).astype(int)[:-1]
        parts = np.split(idx_by_class[c], splits)
        for i, part in enumerate(parts):
            client_indices[i].extend(part.tolist())
    clients = []
    X_all = df[feature_names].values.astype(np.float32)
    for indices in client_indices:
        indices = np.array(indices)
        if len(indices) < 10:  # degenerate: too few samples for a usable shadow split
            indices = rng.choice(len(df), size=max(50, len(indices)), replace=False)
        X, y = X_all[indices], y_all[indices]
        if len(np.unique(y)) > 1 and min(np.bincount(y)) >= 2:
            Xtr, Xsh, ytr, ysh = train_test_split(X, y, test_size=shadow_frac, stratify=y, random_state=seed)
        else:
            Xtr, Xsh, ytr, ysh = train_test_split(X, y, test_size=shadow_frac, random_state=seed)
        clients.append({'X_train': Xtr, 'y_train': ytr, 'X_shadow': Xsh, 'y_shadow': ysh})
    return clients


def exp_noniid_dirichlet(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    alphas = [0.01, 0.1, 0.5, 1.0, 10.0]
    rows = []
    for seed in seeds:
        df_full, feats = get_dataset(dataset, max_rows)
        d = df_full.copy()
        le = LabelEncoder().fit(d['label']); d['label_enc'] = le.transform(d['label'])
        nc = d['label_enc'].nunique()
        dtr, dte = train_test_split(d, test_size=0.2, stratify=d['label_enc'], random_state=seed)
        sc = StandardScaler().fit(dtr[feats]); dtr = dtr.copy(); dte = dte.copy()
        dtr[feats] = sc.transform(dtr[feats]); dte[feats] = sc.transform(dte[feats])
        Xte_ = dte[feats].values.astype(np.float32); yte_ = dte['label_enc'].values
        for alpha in alphas:
            clients = make_clients_dirichlet(dtr, feats, 'label_enc', N_CLIENTS, alpha, seed=seed)
            g, h = run_federated(clients, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=True)
            r = evaluate(len(feats), nc, g, Xte_, yte_)
            fp_rounds = sum(h['n_vetoed'])
            max_client_size = max(len(c['y_train']) for c in clients)
            min_client_size = min(len(c['y_train']) for c in clients)
            rows.append(dict(dataset=dataset, alpha=alpha, seed=seed, clean_f1=r['f1_macro'],
                              fp_twin_rounds=fp_rounds, total_twin_rounds=ROUNDS*len(clients),
                              client_size_ratio=max_client_size/max(min_client_size,1)))
            print(f'{dataset} alpha={alpha} seed={seed}: clean_f1={r["f1_macro"]:.3f} '
                  f'fp={fp_rounds}/{ROUNDS*len(clients)} size_ratio={max_client_size/max(min_client_size,1):.1f}')
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table_noniid_dirichlet.csv', index=False)
    fig, ax = plt.subplots(figsize=(5, 3.5))
    summary = rdf.groupby('alpha')['fp_twin_rounds'].sum() / rdf.groupby('alpha')['total_twin_rounds'].sum()
    ax.semilogx(summary.index, summary.values * 100, 'o-', color='#C44E52')
    ax.set_xlabel('Dirichlet alpha (lower = more non-IID)'); ax.set_ylabel('False-positive rate (%)')
    ax.set_title(f'Does heterogeneity look like an attack? -- {dataset}')
    plt.tight_layout(); plt.savefig(f'{outdir}/noniid_fp_rate.png', dpi=150); plt.close(fig)
    return {'section': 'noniid_dirichlet', 'dataset': dataset, 'summary': summary.to_dict()}

## 19. Client-count scalability

Everything else in this notebook uses $N=8$ (or 9). This sweeps $N \in \{8, 16, 32, 64, 128\}$ (holding total data roughly fixed, so more clients means less data per client, itself a realistic edge scenario) and measures wall-clock time per round, detection under a proportional attack (roughly 1/3 of twins compromised at every scale), and false-positive rate -- the gate's peer-relative statistic is a median/MAD over $N$ scalar values per round, so its own compute cost is negligible regardless of $N$; what matters is whether detection quality holds as $N$ grows and per-client data shrinks.

In [ ]:
def exp_scalability(dataset, outdir):
    seed = SEEDS_QUICK[0]
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    df_full, feats = get_dataset(dataset, max_rows)
    d = df_full.copy()
    le = LabelEncoder().fit(d['label']); d['label_enc'] = le.transform(d['label'])
    nc = d['label_enc'].nunique()
    dtr, dte = train_test_split(d, test_size=0.2, stratify=d['label_enc'], random_state=seed)
    sc = StandardScaler().fit(dtr[feats]); dtr = dtr.copy(); dte = dte.copy()
    dtr[feats] = sc.transform(dtr[feats]); dte[feats] = sc.transform(dte[feats])
    Xte_ = dte[feats].values.astype(np.float32); yte_ = dte['label_enc'].values
    rows = []
    for n_clients in [8, 16, 32, 64, 128]:
        clients = make_clients_dirichlet(dtr, feats, 'label_enc', n_clients, alpha=0.5, seed=seed)
        n_poison = max(1, n_clients // 3)
        poison = copy.deepcopy(clients)
        rng = np.random.default_rng(seed)
        for pc in range(n_poison):
            poison[pc]['y_train'] = rng.integers(0, nc, size=len(poison[pc]['y_train']))
        t0 = time.time()
        g, h = run_federated(poison, len(feats), nc, rounds=ROUNDS, lr=LR, seed=seed, use_twin_gate=True)
        elapsed = time.time() - t0
        r = evaluate(len(feats), nc, g, Xte_, yte_)
        caught = sum(1 for v in h['vetoed_clients'] if any(p < n_poison for p in v))
        rows.append(dict(dataset=dataset, n_clients=n_clients, seconds_total=elapsed,
                          seconds_per_round=elapsed/ROUNDS, f1=r['f1_macro'], caught_rounds=caught,
                          n_poisoned=n_poison))
        print(f'{dataset} N={n_clients}: {elapsed:.0f}s total ({elapsed/ROUNDS:.2f}s/round), '
              f'f1={r["f1_macro"]:.3f}, caught={caught}/{ROUNDS}')
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table_scalability.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    axes[0].plot(rdf['n_clients'], rdf['seconds_per_round'], 'o-', color='#4C72B0')
    axes[0].set_xlabel('Number of twins (N)'); axes[0].set_ylabel('Seconds / round'); axes[0].set_xscale('log', base=2)
    axes[1].plot(rdf['n_clients'], rdf['f1'], 's-', color='#55A868')
    axes[1].set_xlabel('Number of twins (N)'); axes[1].set_ylabel('Attacked macro-F1'); axes[1].set_xscale('log', base=2)
    plt.suptitle(f'Scalability -- {dataset} (~1/3 of twins compromised throughout)')
    plt.tight_layout(); plt.savefig(f'{outdir}/scalability.png', dpi=150); plt.close(fig)
    return {'section': 'scalability', 'dataset': dataset, 'summary': rdf.to_dict('records')}

## 20. Convergence-aware gate: replacing fixed epsilon with a scale-aware threshold

Section 11's long-horizon diagnosis found that loss-delta *magnitude* shrinks as training converges while cross-twin *spread* does not shrink at the same rate, so a fixed absolute threshold ($\varepsilon=0$ or $0.05$) becomes progressively more sensitive to ordinary noise. Rather than patch this with a single constant, this variant makes the absolute threshold track the round's own scale: $\varepsilon_t = c \cdot \mathrm{MAD}(\Delta_t)$ for a small constant $c$, so the threshold shrinks and grows with the same statistic that already governs the peer-relative test, instead of staying fixed while that statistic moves. This directly tests whether that fixes the long-horizon failure more completely than the flat $\varepsilon=0.05$ patch did.

In [ ]:
def our_gate_aggregate_adaptive_eps(candidates, weights, global_state, client_data, in_dim_, nc_,
                                      c=2.0, veto_k=4.0, sigma_min=0.01):
    '''Same peer-relative (MAD) test as the standard gate; the absolute test now uses
    epsilon_t = c * MAD(deltas) instead of a fixed constant.'''
    deltas = []
    for cd, cand in zip(client_data, candidates):
        before = evaluate(in_dim_, nc_, global_state, cd['X_shadow'], cd['y_shadow'])
        after = evaluate(in_dim_, nc_, cand, cd['X_shadow'], cd['y_shadow'])
        deltas.append(after['loss'] - before['loss'])
    deltas = np.array(deltas)
    median_delta = float(np.median(deltas))
    mad = max(float(np.median(np.abs(deltas - median_delta))), sigma_min)
    eps_t = c * mad
    new_states, new_weights, vetoed = [], [], []
    for i in range(len(candidates)):
        veto = (deltas[i] > eps_t) or (deltas[i] > median_delta + veto_k * mad)
        if veto: vetoed.append(i)
        else: new_states.append(candidates[i]); new_weights.append(weights[i])
    new_global = average_states(new_states, new_weights) if new_states else global_state
    return new_global, deltas, vetoed, eps_t


def run_federated_long_adaptive(clients, in_dim_, nc_, rounds, seed, c=2.0):
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    cum_vetoed = 0; trace = []; eps_trace = []
    for r in range(1, rounds + 1):
        candidates, weights = [], []
        for cd in clients:
            cw = compute_class_weights(cd['y_train'], nc_)
            cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed, eps_t = our_gate_aggregate_adaptive_eps(candidates, weights, gstate, clients,
                                                                        in_dim_, nc_, c=c, veto_k=VETO_K)
        cum_vetoed += len(vetoed)
        eps_trace.append(eps_t)
        if r in (15, 30, 50): trace.append((r, cum_vetoed))
    return cum_vetoed, trace, eps_trace


def exp_convergence_aware_gate(dataset, outdir, seed=1, c=2.0):
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
    # clean-condition long-horizon comparison: fixed eps=0 vs fixed eps=0.05 vs adaptive eps_t=c*MAD_t
    cum_fixed0, trace_fixed0 = run_federated_long(clients, len(feats), nc, 50, seed, veto_epsilon=0.0)
    cum_fixed05, trace_fixed05 = run_federated_long(clients, len(feats), nc, 50, seed, veto_epsilon=0.05)
    cum_adaptive, trace_adaptive, eps_trace = run_federated_long_adaptive(clients, len(feats), nc, 50, seed, c=c)
    # also check detection is not lost at the standard 15-round horizon under a real attack
    n_poison = max(1, len(clients) // 3)
    poison = copy.deepcopy(clients)
    rng = np.random.default_rng(seed)
    for pc in range(n_poison):
        poison[pc]['y_train'] = rng.integers(0, nc, size=len(poison[pc]['y_train']))
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(len(feats), nc))
    caught_adaptive = 0
    for r in range(ROUNDS):
        candidates, weights = [], []
        for cd in poison:
            cw = compute_class_weights(cd['y_train'], nc)
            cand = local_train(len(feats), nc, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed, eps_t = our_gate_aggregate_adaptive_eps(candidates, weights, gstate, poison,
                                                                        len(feats), nc, c=c, veto_k=VETO_K)
        if any(p in vetoed for p in range(n_poison)): caught_adaptive += 1
    r15 = evaluate(len(feats), nc, gstate, Xte, yte)
    out = {'dataset': dataset, 'seed': seed, 'c': c,
           'trace_fixed_eps0': trace_fixed0, 'trace_fixed_eps005': trace_fixed05,
           'trace_adaptive': trace_adaptive, 'eps_t_range': [min(eps_trace), max(eps_trace)],
           'detection_at_15_adaptive': {'caught_rounds': caught_adaptive, 'f1': r15['f1_macro']}}
    with open(f'{outdir}/convergence_aware_gate.json', 'w') as f:
        json.dump(out, f, indent=2, default=str)
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for trace, label, color in [(trace_fixed0, 'Fixed eps=0', '#C44E52'),
                                  (trace_fixed05, 'Fixed eps=0.05', '#DD8452'),
                                  (trace_adaptive, f'Adaptive eps_t={c}*MAD_t', '#55A868')]:
        rounds_x = [t[0] for t in trace]; cum_y = [t[1] for t in trace]
        ax.plot(rounds_x, cum_y, 'o-', label=label, color=color)
    ax.set_xlabel('Round'); ax.set_ylabel('Cumulative false vetoes')
    ax.set_title(f'Convergence-aware threshold -- {dataset}'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f'{outdir}/convergence_aware_gate.png', dpi=150); plt.close(fig)
    return {'section': 'convergence_aware_gate', 'dataset': dataset, 'summary': out}

## 21. Ablation extended to the stronger (parameter-space) attacks
Section 8's ablation (absolute-only / MAD-only / combined) covered label-flipping attacks. This repeats it for scaling, sign-flipping, and model-replacement -- the parameter-space attacks from Section 16 -- to check whether the same component breakdown holds for a qualitatively different attack family.

In [ ]:
def run_gate_variant_strong(clients, in_dim_, nc_, attack_type, ablation, seed, lam=10.0, attacker_idx=0):
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(in_dim_, nc_))
    caught = 0
    for r in range(ROUNDS):
        candidates, weights = [], []
        for i, cd in enumerate(clients):
            cw = compute_class_weights(cd['y_train'], nc_)
            honest_cand = local_train(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            if i == attacker_idx:
                if attack_type == 'scaling': cand = poison_scaling(honest_cand, gstate, lam=lam)
                elif attack_type == 'sign_flip': cand = poison_sign_flip(honest_cand, gstate, lam=lam)
                else: cand = poison_model_replacement(in_dim_, nc_, gstate, cd['X_train'], cd['y_train'],
                                                       LR, target_class=0, n_twins=len(clients))
            else:
                cand = honest_cand
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
        gstate, deltas, vetoed = our_gate_aggregate(candidates, weights, gstate, clients, in_dim_, nc_,
                                                     VETO_EPSILON, VETO_K, ablation=ablation)
        if attacker_idx in vetoed: caught += 1
    return caught


def exp_ablation_strong(dataset, outdir):
    seeds = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    rows = []
    for seed in seeds:
        clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
        for attack_type in ['scaling', 'sign_flip', 'model_replacement']:
            for ablation_name in ['absolute_only', 'mad_only', None]:
                label = ablation_name or 'combined'
                caught = run_gate_variant_strong(clients, len(feats), nc, attack_type, ablation_name, seed)
                rows.append(dict(dataset=dataset, attack=attack_type, variant=label, seed=seed, caught=caught))
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{outdir}/table_ablation_strong_attacks.csv', index=False)
    return {'section': 'ablation_strong', 'dataset': dataset,
            'summary': rdf.groupby(['attack','variant'])['caught'].mean().to_dict()}

## 22. Z-score separation: visualizing the theoretical argument
Directly visualizes the statistic the peer-relative test thresholds on: $Z_i = (\Delta_i - \mathrm{median}(\Delta)) / \max(\mathrm{MAD}(\Delta), \sigma_{\min})$, computed every round for every twin, honest and compromised, and pooled across rounds. This is the empirical counterpart to the Cantelli-inequality argument in the paper: under a bounded-variance assumption, $P(Z_i \geq k) \leq 1/(1+k^2)$ for **any** distribution, regardless of shape -- this figure shows what the honest- and malicious-twin $Z$ distributions actually look like on real data, against the $k=4$ line the gate uses.

In [ ]:
def exp_zscore_distribution(dataset, outdir, seed=1):
    max_rows = MAX_ROWS_QUICK if QUICK_MODE else MAX_ROWS_FULL
    clients, feats, nc, Xte, yte, le = get_clients_for_seed(dataset, seed, max_rows)
    n_poison = max(1, len(clients) // 3)
    poison = copy.deepcopy(clients)
    rng = np.random.default_rng(seed)
    for pc in range(n_poison):
        poison[pc]['y_train'] = rng.integers(0, nc, size=len(poison[pc]['y_train']))
    torch.manual_seed(seed); np.random.seed(seed)
    gstate = get_state(TwinMLP(len(feats), nc))
    honest_z, malicious_z = [], []
    for r in range(ROUNDS):
        candidates, weights, deltas = [], [], []
        for cd in poison:
            cw = compute_class_weights(cd['y_train'], nc)
            cand = local_train(len(feats), nc, gstate, cd['X_train'], cd['y_train'], lr=LR, class_weights=cw)
            before = evaluate(len(feats), nc, gstate, cd['X_shadow'], cd['y_shadow'])
            after = evaluate(len(feats), nc, cand, cd['X_shadow'], cd['y_shadow'])
            candidates.append(cand); weights.append(cd['X_train'].shape[0])
            deltas.append(after['loss'] - before['loss'])
        deltas = np.array(deltas)
        med = np.median(deltas); mad = max(np.median(np.abs(deltas - med)), 0.01)
        z = (deltas - med) / mad
        for i in range(len(poison)):
            (malicious_z if i < n_poison else honest_z).append(z[i])
        new_states, new_weights = [], []
        for i in range(len(poison)):
            veto = deltas[i] > 0 or deltas[i] > med + VETO_K * mad
            if not veto: new_states.append(candidates[i]); new_weights.append(weights[i])
        if new_states: gstate = average_states(new_states, new_weights)
    pd.DataFrame({'z': honest_z + malicious_z, 'type': ['honest']*len(honest_z) + ['malicious']*len(malicious_z)}
                 ).to_csv(f'{outdir}/zscore_distribution.csv', index=False)
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    bins = np.linspace(min(honest_z+malicious_z+[-1]), max(honest_z+malicious_z+[1]), 30)
    ax.hist(honest_z, bins=bins, alpha=0.6, label='Honest twins', color='#55A868', density=True)
    ax.hist(malicious_z, bins=bins, alpha=0.6, label='Compromised twins', color='#C44E52', density=True)
    ax.axvline(VETO_K, color='black', linestyle='--', label=f'k={VETO_K} threshold')
    ax.set_xlabel('Z-score'); ax.set_ylabel('Density'); ax.legend(fontsize=8)
    ax.set_title(f'Honest vs. compromised twin Z-scores -- {dataset}')
    plt.tight_layout(); plt.savefig(f'{outdir}/zscore_distribution.png', dpi=150); plt.close(fig)
    return {'section': 'zscore_distribution', 'dataset': dataset,
            'summary': {'honest_mean_z': float(np.mean(honest_z)), 'honest_max_z': float(np.max(honest_z)),
                        'malicious_mean_z': float(np.mean(malicious_z)), 'malicious_min_z': float(np.min(malicious_z)),
                        'frac_honest_above_k': float(np.mean(np.array(honest_z) >= VETO_K)),
                        'frac_malicious_above_k': float(np.mean(np.array(malicious_z) >= VETO_K))}}

## 23. Orchestration: run every experiment for one dataset, with error isolation
Each experiment runs in its own try/except. A failure (e.g.\ a real column name that doesn't match, or a transient error) is caught, logged with its full traceback to `run_log.txt`, and the loop moves on to the next experiment -- it does not abort the dataset or the run. `run_dataset` itself is also wrapped by the caller, so a total load failure for one dataset (e.g.\ Edge-IIoTset's `Attack_type` column not existing) still lets the other two datasets run.

In [ ]:
RUN_LOG = []

def log_event(dataset, section, status, detail=''):
    entry = {'dataset': dataset, 'section': section, 'status': status, 'detail': detail,
             'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    RUN_LOG.append(entry)
    print(f"[{entry['timestamp']}] {dataset}/{section}: {status}" + (f' -- {detail[:200]}' if detail else ''))


def run_experiment(fn, dataset, outdir, *args, **kwargs):
    section = fn.__name__.replace('exp_', '')
    try:
        t0 = time.time()
        result = fn(dataset, outdir, *args, **kwargs)
        log_event(dataset, section, 'OK', f'{time.time()-t0:.0f}s')
        return result
    except Exception as e:
        tb = traceback.format_exc()
        log_event(dataset, section, 'FAILED', tb)
        with open(f'{outdir}/ERROR_{section}.txt', 'w') as f:
            f.write(tb)
        return None


def run_dataset(dataset):
    outdir = os.path.join(RESULTS_DIR, dataset)
    os.makedirs(outdir, exist_ok=True)
    print(f'\n{"="*70}\n{dataset.upper()}\n{"="*70}')

    if not ensure_downloaded(dataset):
        log_event(dataset, 'download', 'FAILED', 'no CSVs found after download attempt')
        return

    try:
        df_probe, feats_probe = get_dataset(dataset, MAX_ROWS_FULL)
        with open(f'{outdir}/data_summary.json', 'w') as f:
            json.dump({'shape': list(df_probe.shape), 'label_counts': df_probe['label'].value_counts().to_dict(),
                       'n_features': len(feats_probe), 'device_id_sample': df_probe['device_id'].unique()[:5].tolist()},
                      f, indent=2)
        log_event(dataset, 'initial_load', 'OK', f'{df_probe.shape}')
    except Exception as e:
        log_event(dataset, 'initial_load', 'FAILED', traceback.format_exc())
        return  # can't proceed with this dataset at all

    all_summaries = []
    all_summaries.append(run_experiment(exp_single_attacker, dataset, outdir))
    all_summaries.append(run_experiment(exp_baseline_comparison, dataset, outdir))
    all_summaries.append(run_experiment(exp_ablation, dataset, outdir))
    all_summaries.append(run_experiment(exp_adaptive_attacker, dataset, outdir))
    all_summaries.append(run_experiment(exp_sensitivity, dataset, outdir))
    all_summaries.append(run_experiment(exp_long_horizon, dataset, outdir))
    all_summaries.append(run_experiment(exp_generalization, dataset, outdir))
    all_summaries.append(run_experiment(exp_severity_sweep, dataset, outdir))
    all_summaries.append(run_experiment(exp_shap, dataset, outdir))
    all_summaries.append(run_experiment(exp_fairness, dataset, outdir))
    all_summaries.append(run_experiment(exp_strong_attacks, dataset, outdir))
    all_summaries.append(run_experiment(exp_peer_aware_attacker, dataset, outdir))
    all_summaries.append(run_experiment(exp_noniid_dirichlet, dataset, outdir))
    all_summaries.append(run_experiment(exp_scalability, dataset, outdir))
    all_summaries.append(run_experiment(exp_convergence_aware_gate, dataset, outdir))
    all_summaries.append(run_experiment(exp_ablation_strong, dataset, outdir))
    all_summaries.append(run_experiment(exp_zscore_distribution, dataset, outdir))

    with open(f'{outdir}/all_summaries.json', 'w') as f:
        json.dump([s for s in all_summaries if s is not None], f, indent=2, default=str)
    print(f'{dataset}: {sum(1 for s in all_summaries if s is not None)}/{len(all_summaries)} sections succeeded')

## 16b. Sanity check before the long run
Verifies every config global and function this notebook depends on actually exists in this kernel session, **before** committing to a multi-hour run. This is what would have caught the `NameError: name 'CICIOT_DATA_DIR' is not defined` failure immediately instead of three times over in the main loop -- if you see anything other than `All checks passed` below, do `Runtime > Restart runtime` then `Runtime > Run all` and re-check before proceeding.

In [ ]:
_required_names = ['CICIOT_DATA_DIR', 'EDGEIIOT_DATA_DIR', 'NBAIOT_DATA_DIR', 'DATA_DIRS', 'KAGGLE_SLUGS',
                    'DATASETS', 'RESULTS_DIR', 'QUICK_MODE', 'MAX_ROWS_FULL', 'MAX_ROWS_QUICK',
                    'SEEDS_FULL', 'SEEDS_QUICK', 'N_CLIENTS', 'ROUNDS', 'LOCAL_EPOCHS', 'LR', 'VETO_K',
                    'WEIGHT_CAP', 'load_real_ciciot', 'load_real_edgeiiot', 'load_real_nbaiot',
                    'ensure_downloaded', 'get_dataset', 'get_clients_for_seed', 'TwinMLP', 'run_federated',
                    'our_gate_aggregate', 'run_federated_method', 'exp_single_attacker', 'exp_baseline_comparison',
                    'exp_ablation', 'exp_adaptive_attacker', 'exp_sensitivity', 'exp_long_horizon',
                    'exp_generalization', 'exp_severity_sweep', 'exp_shap', 'exp_fairness', 'run_dataset']
missing = [n for n in _required_names if n not in globals()]
if missing:
    print('!'*70)
    print('MISSING from this kernel session:', missing)
    print('This almost always means not every cell above has been run in order in the')
    print('CURRENT session (e.g. after a runtime restart or disconnect). Fix: Runtime >')
    print('Restart runtime, then Runtime > Run all. Re-run this cell to confirm before')
    print('proceeding to Section 17.')
    print('!'*70)
else:
    print(f'All checks passed ({len(_required_names)} names found). Safe to proceed to Section 17.')

## 24. Run everything
This is the one cell that does it all. It will take a long time -- that's expected. Progress prints as it goes; you don't need to watch it.

In [ ]:
t_start = time.time()
for dataset in DATASETS:
    try:
        run_dataset(dataset)
    except Exception as e:
        log_event(dataset, 'run_dataset', 'FATAL', traceback.format_exc())
        print(f'{dataset}: FATAL error, moving to next dataset. See run_log.txt for details.')

with open(os.path.join(RESULTS_DIR, 'run_log.json'), 'w') as f:
    json.dump(RUN_LOG, f, indent=2)
with open(os.path.join(RESULTS_DIR, 'run_log.txt'), 'w') as f:
    for e in RUN_LOG:
        f.write(f"[{e['timestamp']}] {e['dataset']}/{e['section']}: {e['status']}\n")
        if e['detail']:
            f.write(f"    {e['detail']}\n")

n_ok = sum(1 for e in RUN_LOG if e['status']=='OK')
n_fail = sum(1 for e in RUN_LOG if e['status'] in ('FAILED','FATAL'))
print(f'\n\nALL DONE in {(time.time()-t_start)/60:.1f} minutes.')
print(f'Succeeded: {n_ok}')
print(f'Failed:    {n_fail}')

if n_ok == 0:
    print('\n' + '!'*70)
    print('WARNING: nothing succeeded at all. This almost always means the data')
    print('download step failed for every dataset (check the kaggle CLI verification')
    print('cell above, and read results/run_log.txt for the exact error). The zip')
    print('below will NOT contain result tables or images if this is the case.')
    print('!'*70)
elif any(e['section'] == 'download' and e['status'] == 'FAILED' for e in RUN_LOG):
    failed_dl = [e['dataset'] for e in RUN_LOG if e['section']=='download' and e['status']=='FAILED']
    print(f'\nNOTE: download failed for: {failed_dl} -- those datasets have no results. See run_log.txt.')

## 25. Bundle everything into one zip and download it
This is the file to send back. It contains every CSV/JSON/PNG from every experiment that succeeded for every dataset, plus `run_log.txt` showing exactly what ran, what failed, and why.

In [ ]:
zip_path = 'results_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(RESULTS_DIR):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, RESULTS_DIR))

print(f'{zip_path} created, {os.path.getsize(zip_path)/1e6:.1f} MB')
print('Contents:')
with zipfile.ZipFile(zip_path) as zf:
    for n in sorted(zf.namelist()):
        print(' ', n)

In [ ]:
from google.colab import files
files.download(zip_path)